# NLP Feature Engineering and Sentiment Classification

In [7]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

In [8]:
df = pd.read_csv('../data/cleaned_reviews.csv')
# Combine title and description
df['text'] = df['Review Title Cleaned'].fillna('') + ' ' + df['Review Description Cleaned'].fillna('')
# Extract star rating
df['Star Rating'] = df['Star Rating'].str.extract(r'(\d+)').astype(int)
# Create sentiment: positive if Star Rating >=4, negative if <=2
df = df[df['Star Rating'] != 3]
df['sentiment'] = df['Star Rating'].apply(lambda x: 1 if x >= 4 else 0)
texts = df['text'].tolist()
labels = df['sentiment'].tolist()
print(f"Total reviews: {len(texts)}")
print(f"Positive: {sum(labels)}, Negative: {len(labels) - sum(labels)}")

Total reviews: 656
Positive: 586, Negative: 70


## Task 2: Vocabulary Creation

In [9]:
vectorizer = CountVectorizer()
X_bow = vectorizer.fit_transform(texts[:1000])  # use subset for speed
vocab = vectorizer.get_feature_names_out()
print(f"Vocabulary size: {len(vocab)}")
# Top frequent words
word_counts = X_bow.sum(axis=0).A1
word_freq = dict(zip(vocab, word_counts))
sorted_words = sorted(word_freq.items(), key=lambda x: x[1], reverse=True)
print("Top 10 frequent words:")
for word, freq in sorted_words[:10]:
    print(f"{word}: {freq}")

Vocabulary size: 4801
Top 10 frequent words:
good: 783
more: 776
quality: 627
sound: 580
battery: 452
earbuds: 449
very: 370
are: 362
but: 344
you: 318


## Task 3: Feature Engineering

### 1. One Hot Encoding (document-level vector)

In [10]:
# For OHE, since vocab is large, do for small subset
small_texts = texts[:10]
small_vectorizer = CountVectorizer(binary=True)
X_ohe = small_vectorizer.fit_transform(small_texts)
print("OHE matrix shape:", X_ohe.shape)
print("OHE for first document:", X_ohe[0].toarray())

OHE matrix shape: (10, 290)
OHE for first document: [[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 1 1 0 0
  0 0 0 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 1 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 1 0 1
  0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1
  0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  1 0 0 0 0 0 0 0 1 0 1 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 1 0 0 1 0 0 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  1 0]]


### 2. Bag of Words using CountVectorizer

In [11]:
bow_vectorizer = CountVectorizer()
X_bow_full = bow_vectorizer.fit_transform(texts)
print("BoW matrix shape:", X_bow_full.shape)

BoW matrix shape: (656, 4801)


### 3. TF-IDF using TfidfVectorizer

In [12]:
tfidf_vectorizer = TfidfVectorizer()
X_tfidf = tfidf_vectorizer.fit_transform(texts)
print("TF-IDF matrix shape:", X_tfidf.shape)

TF-IDF matrix shape: (656, 4801)


## Task 4: Comparison Analysis

In [13]:
# For comparison, use small subset
small_texts = texts[:5]
bow_small = CountVectorizer()
X_bow_small = bow_small.fit_transform(small_texts)
tfidf_small = TfidfVectorizer()
X_tfidf_small = tfidf_small.fit_transform(small_texts)
ohe_small = CountVectorizer(binary=True)
X_ohe_small = ohe_small.fit_transform(small_texts)
# Print shapes
print("OHE shape:", X_ohe_small.shape)
print("BoW shape:", X_bow_small.shape)
print("TF-IDF shape:", X_tfidf_small.shape)
# Explain
print("\nExplanation: In TF-IDF, important words are those with high TF-IDF scores, which are frequent in a document but rare across documents. Common words get lower weights because IDF is low.")

OHE shape: (5, 141)
BoW shape: (5, 141)
TF-IDF shape: (5, 141)

Explanation: In TF-IDF, important words are those with high TF-IDF scores, which are frequent in a document but rare across documents. Common words get lower weights because IDF is low.


## Task 5: Sparse Matrix Analysis

In [14]:
def calculate_sparsity(matrix):
    total_elements = matrix.shape[0] * matrix.shape[1]
    non_zero = matrix.nnz
    sparsity = (1 - non_zero / total_elements) * 100
    return sparsity
print("BoW sparsity:", calculate_sparsity(X_bow_full))
print("TF-IDF sparsity:", calculate_sparsity(X_tfidf))
print("Explanation: Sparse matrices have many zeros, inefficient for storage and computation in large-scale systems because they waste memory and processing on zeros.")

BoW sparsity: 98.89755564135521
TF-IDF sparsity: 98.89755564135521
Explanation: Sparse matrices have many zeros, inefficient for storage and computation in large-scale systems because they waste memory and processing on zeros.


## Task 6: Real-world Questions

1. Why Bag of Words fails in understanding semantic meaning (example: similar meaning words)

Bag of Words treats words as independent units without considering meaning or context. For example, "good" and "excellent" have similar meanings but are treated as different features, so semantic similarity is not captured.

2. When to use Bag of Words and TF-IDF in industry

Use BoW for simple text classification where word frequency matters. Use TF-IDF when you want to downweight common words and highlight important terms, common in information retrieval and document classification.

3. Limitations of TF-IDF in real applications

TF-IDF doesn't capture word order, context, or semantics. It can be affected by document length, and doesn't handle synonyms or polysemy.

## Task 7: Mini Use Case

In [15]:
# Split data
X_train_bow, X_test_bow, y_train, y_test = train_test_split(X_bow_full, labels, test_size=0.2, random_state=42)
X_train_tfidf, X_test_tfidf, y_train_tfidf, y_test_tfidf = train_test_split(X_tfidf, labels, test_size=0.2, random_state=42)
# Logistic Regression with BoW
lr_bow = LogisticRegression()
lr_bow.fit(X_train_bow, y_train)
pred_bow = lr_bow.predict(X_test_bow)
print("BoW Logistic Regression Accuracy:", accuracy_score(y_test, pred_bow))
# With TF-IDF
lr_tfidf = LogisticRegression()
lr_tfidf.fit(X_train_tfidf, y_train_tfidf)
pred_tfidf = lr_tfidf.predict(X_test_tfidf)
print("TF-IDF Logistic Regression Accuracy:", accuracy_score(y_test_tfidf, pred_tfidf))
# Alternatively, Naive Bayes
nb_bow = MultinomialNB()
nb_bow.fit(X_train_bow, y_train)
pred_nb_bow = nb_bow.predict(X_test_bow)
print("BoW Naive Bayes Accuracy:", accuracy_score(y_test, pred_nb_bow))
nb_tfidf = MultinomialNB()
nb_tfidf.fit(X_train_tfidf, y_train_tfidf)
pred_nb_tfidf = nb_tfidf.predict(X_test_tfidf)
print("TF-IDF Naive Bayes Accuracy:", accuracy_score(y_test_tfidf, pred_nb_tfidf))

BoW Logistic Regression Accuracy: 0.9621212121212122
TF-IDF Logistic Regression Accuracy: 0.8787878787878788
BoW Naive Bayes Accuracy: 0.9318181818181818
TF-IDF Naive Bayes Accuracy: 0.8787878787878788
